# Tutorial 2: Bringing in your own data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gopalam/PyEGRET/blob/main/tutorials/02_your_own_data.ipynb)

Most of the work in a WRTDS analysis happens before the model is fitted:
getting the discharge record and the water-quality results into the right
form, handling values reported below a detection limit, getting the units
right, and checking that the two records actually line up. This notebook
walks through each of those steps with small example files, and then with
the Choptank River files from Tutorial 1, so that when you come to your own
data you know what PyEGRET expects and what it will tell you when something
is off.

You will need two files for each site and constituent:

1. a **daily discharge** file: one row per day, a date and a discharge; and
2. a **water-quality** file: one row per sample, a date, a remark code and a
   concentration.

Both are plain text with comma-separated columns (CSV), which any
spreadsheet can save. If your data are in USGS databases, you can skip the
files entirely and download them; that is the subject of a later tutorial.

In [1]:
import os
import sys

# If this notebook sits in PyEGRET/tutorials, PyEGRET is one folder up and is
# used straight from there, with nothing installed. Anywhere else (Google
# Colab, say) it is installed from GitHub, and the data are read from GitHub.
HERE = os.path.abspath("")
ROOT = os.path.dirname(HERE)
if os.path.isdir(os.path.join(ROOT, "pyegret")):
    sys.path.insert(0, ROOT)
else:
    try:
        import pyegret
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                        "pyegret @ git+https://github.com/gopalam/PyEGRET"], check=True)

# The example data: the data folder next to this notebook, or else GitHub.
DATA = os.path.join(HERE, "data")
if not os.path.isdir(DATA):
    DATA = "https://raw.githubusercontent.com/gopalam/PyEGRET/main/tutorials/data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyegret

print("PyEGRET", pyegret.__version__)

PyEGRET 0.1.0


The examples below write a few small files of their own. They go in a
temporary folder, so nothing is left behind in your project. The cell also
makes Python's warnings print as a single plain line, which is easier to read
here.

In [2]:
import tempfile
import warnings

WORK = tempfile.mkdtemp()


def short_warning(message, category, filename, lineno, line=None):
    """Print warnings as one plain line, without the file they came from."""
    return f"{category.__name__}: {message}\n"


warnings.formatwarning = short_warning


def write_file(name, text):
    """Write a small example file into the temporary folder and return its path."""
    path = os.path.join(WORK, name)
    with open(path, "w") as handle:
        handle.write(text)
    return path

## 1. The daily discharge file

The first column is the date and the second is the mean discharge for that
day. The column names do not matter, only their order. This is the start of
the Choptank file:

In [3]:
pd.read_csv(DATA + "/choptank_daily.csv", nrows=5)

,date,discharge_cfs
0,1979-10-01,67
1,1979-10-02,71
2,1979-10-03,97
3,1979-10-04,136
4,1979-10-05,144


`read_user_daily` reads it and adds everything else WRTDS needs: the log of
discharge, decimal year, day of the year, water year, and 7- and 30-day
running means.

**Units.** As in R EGRET, `read_user_daily` assumes the discharge is in
**cubic feet per second**, the unit USGS publishes, and converts it to cubic
meters per second, which is what PyEGRET works in internally. It prints a
reminder every time. If your file is already in cubic meters per second, say
so with `q_unit=2`; getting this wrong scales every flux by a factor of 35.

**Dates.** ISO dates (`2001-03-15`) and US-style dates (`03/15/2001`) are
recognized without being told. For anything else, pass the format with
`date_format`, using Python's `strftime` codes. Here is a file with European
dates, semicolons between columns and discharge already in m3/s:

In [4]:
from pyegret.data.read_user import read_user_daily, read_user_sample

path = write_file("daily_example.csv", """Date;Flow_m3s
15.03.2001;2.5
16.03.2001;0
18.03.2001;3.1
""")

daily_example = read_user_daily(path, separator=";", q_unit=2, date_format="%d.%m.%Y")
daily_example[["Date", "Q", "LogQ", "DecYear", "waterYear"]]

There are 1 zero flow days and no negative flow days.
All days had 0.0019 cms added to the discharge value.
Rolling means not calculated because there are fewer than 30 days.
25% missing data.
Many EGRET functions will not work with missing values.


,Date,Q,LogQ,DecYear,waterYear
0,2001-03-15,2.501867,0.917037,2001.200000,2001
1,2001-03-16,0.001867,-6.283601,2001.202740,2001
2,2001-03-18,3.101867,1.132004,2001.208219,2001


Two things happened that are worth knowing about, both copied from R:

- **Zero flows.** WRTDS works with the logarithm of discharge, and the log of
  zero does not exist. When a record contains zero (or negative) flows, every
  day's discharge is raised by a small amount, one thousandth of the mean
  discharge, and a message says so. If your stream goes dry for long
  periods, think about whether WRTDS suits it at all.
- **Missing days.** March 17 is not in the file. The record should be
  continuous, and the reader reports the share of days that is missing.
  Many EGRET functions do not work with gaps, so fill short ones (with
  estimated values, as USGS does) before reading the file.

The note about rolling means is only because this example is three days
long; they need at least 30.

## 2. The water-quality file

The first column is the sample date. Then come pairs of columns: a **remark
code** and a **value**. The remark is left blank for a measured value, `<`
for a value below the reporting limit (in which case the value is the
limit), and `>` for a value above an upper limit.

In [5]:
pd.read_csv(DATA + "/choptank_sample.csv", keep_default_na=False, nrows=5)

,date,remark,nitrate_nitrite_mgL
0,1979-10-24,,0.62
1,1979-12-05,,1.40
2,1979-12-21,,1.20
3,1980-01-24,,0.84
4,1980-02-25,,1.20


### Censored values

A result reported as "< 0.05" says only that the true concentration lies
somewhere between zero and 0.05. PyEGRET keeps exactly that information: each
sample becomes an interval with a lower bound `ConcLow` and an upper bound
`ConcHigh`, and `Uncen` is 1 for a measured value and 0 for a censored one.
The regression uses the full likelihood of the interval, as R EGRET does
with `survreg`, so censored values count in the fit; they are never dropped
or replaced with half the limit. (Where R stores the lower bound of a
"less than" value as `NA`, PyEGRET stores 0. They mean the same thing.)

Here is the one censored sample in the Choptank record:

In [6]:
sample = read_user_sample(DATA + "/choptank_sample.csv")
censored = sample["Uncen"] == 0
sample[censored][["Date", "ConcLow", "ConcHigh", "Uncen", "ConcAve"]]

,Date,ConcLow,ConcHigh,Uncen,ConcAve
381,1998-12-14,0.0,0.05,0,0.025


`ConcAve`, the middle of the interval, is used only for plotting and simple
summaries.

### Constituents reported in parts

Sometimes the quantity you want is the sum of several reported values: nitrate
plus nitrite, or dissolved plus particulate phosphorus. Put each part in its
own remark and value pair and `read_user_sample` adds them up, keeping track
of censoring, as EGRET does. If one part is "less than", the sum becomes an
interval too:

In [7]:
path = write_file("parts_example.csv", """date,rNO3,NO3,rNO2,NO2
2001-03-15,,1.20,<,0.01
2001-04-12,,0.95,,0.02
2001-05-10,<,0.05,<,0.01
""")

parts = read_user_sample(path)
parts[["Date", "ConcLow", "ConcHigh", "Uncen"]]

,Date,ConcLow,ConcHigh,Uncen
0,2001-03-15,1.20,1.21,0
1,2001-04-12,0.97,0.97,1
2,2001-05-10,0.00,0.06,0


In March, nitrate was measured at 1.20 and nitrite was below 0.01, so the sum
lies between 1.20 and 1.21. In April both were measured, so the sum is exact.
In May both were below their limits, so the sum lies between 0 and 0.06.

### Concentration units

PyEGRET assumes concentrations in **milligrams per liter**, because flux is
computed from concentration times discharge. Results in micrograms per liter
must be divided by 1000 before they go in the file, or every flux will be a
thousand times too large. Concentration in other units, such as specific
conductance, can still be modelled, but the flux results will then be
meaningless.

## 3. From a laboratory export to a PyEGRET file

Water-quality results rarely arrive in this format. A typical laboratory or
database export has one row per result, with the value, the units, a
qualifier code and a reporting limit. Below is a made-up example of total
phosphorus in micrograms per liter, where the laboratory marks non-detects
with the qualifier `U` and estimated values with `J`. (Qualifier codes differ
between laboratories and databases, so check what yours use.)

In [8]:
lab = pd.DataFrame({
    "SampleDate": ["2021-01-12", "2021-02-09", "2021-03-09", "2021-04-13", "2021-05-11"],
    "Analyte": "Total Phosphorus",
    "Result": [48.0, 31.0, 5.0, 22.0, 17.0],
    "Units": "ug/L",
    "Qualifier": ["", "J", "U", "", ""],
    "ReportingLimit": [5.0, 5.0, 5.0, 5.0, 5.0],
})
lab

,SampleDate,Analyte,Result,Units,Qualifier,ReportingLimit
0,2021-01-12,Total Phosphorus,48.0,ug/L,,5.0
1,2021-02-09,Total Phosphorus,31.0,ug/L,J,5.0
2,2021-03-09,Total Phosphorus,5.0,ug/L,U,5.0
3,2021-04-13,Total Phosphorus,22.0,ug/L,,5.0
4,2021-05-11,Total Phosphorus,17.0,ug/L,,5.0


Three changes turn it into a PyEGRET file. A non-detect gets the remark `<`
and, as its value, the reporting limit. Every value is converted to mg/L. And
the columns go in the expected order: date, remark, value. An estimated
(`J`) value is still a measurement, so it stays uncensored.

In [9]:
non_detect = lab["Qualifier"] == "U"
remark = np.where(non_detect, "<", "")
value_ugL = np.where(non_detect, lab["ReportingLimit"], lab["Result"])

tp = pd.DataFrame({
    "date": lab["SampleDate"],
    "remark": remark,
    "tp_mgL": value_ugL / 1000.0,
})
tp_path = os.path.join(WORK, "tp_sample.csv")
tp.to_csv(tp_path, index=False)

tp_sample = read_user_sample(tp_path)
tp_sample[["Date", "ConcLow", "ConcHigh", "Uncen"]]

,Date,ConcLow,ConcHigh,Uncen
0,2021-01-12,0.048,0.048,1
1,2021-02-09,0.031,0.031,1
2,2021-03-09,0.000,0.005,0
3,2021-04-13,0.022,0.022,1
4,2021-05-11,0.017,0.017,1


The same approach works for a spreadsheet: read it with `pd.read_excel`, build
the three columns as above, and save them with `to_csv`. Keeping that small
conversion step in a script, rather than editing the spreadsheet by hand,
means it can be rerun when new results arrive.

## 4. Site information

The third piece is `INFO`, a one-row table describing the site and the
constituent. Only two entries are needed: `shortName`, the site name that
appears in table and plot titles, and `paramShortName`, the constituent.
`station_nm` (the full station name) and `drainSqKm` (drainage area in
square kilometers, used by a few outputs) are worth adding. Anything else you
add is carried along untouched.

In [10]:
info = pd.DataFrame([{
    "station_nm": "Choptank River near Greensboro, MD",
    "shortName": "Choptank River",
    "paramShortName": "Inorganic nitrogen (nitrate and nitrite)",
    "drainSqKm": 292.67,
}])

## 5. Putting the pieces together, and checking them

`merge_report` joins the three into an eList and attaches each sample's
discharge, matching by date. `data_overview` then summarizes the record and
lists anything that would trouble the model.

In [11]:
from pyegret import merge_report
from pyegret.data.quality import data_overview

daily = read_user_daily(DATA + "/choptank_daily.csv", verbose=False)
elist = merge_report(info, daily, sample)
overview = data_overview(elist)

Discharge record: 11688 days (32.0 years), 1979-10-01 to 2011-09-30
  Q (min / mean / max): 0.009911 / 4.087 / 246.4
Water-quality record: 606 samples (18.9 per year)
  1979-10-24 to 2011-09-29
  concentration (min / mean / max): 0.05 / 1.14 / 2.43
  censored: 1 of 606 (0.2%)


No problems are listed, so this record is ready to model. `data_overview`
also returns everything it printed as a dictionary, so you can run it over
every site in a monitoring network and collect the results in one table.

It is worth seeing what a problem looks like. The most common one is a
discharge record that starts later (or ends earlier) than the sampling.
Here the discharge record is cut to start in 1985, while sampling began in
1979:

In [12]:
late_start = daily["Date"] >= "1985-01-01"
daily_short = daily[late_start]
daily_short = daily_short.reset_index(drop=True)

elist_short = merge_report(info, daily_short, sample)
overview_short = data_overview(elist_short)

Discharge record: 9769 days (26.7 years), 1985-01-01 to 2011-09-30
  Q (min / mean / max): 0.009911 / 4.125 / 246.4
Water-quality record: 606 samples (22.7 per year)
  1979-10-24 to 2011-09-29
  concentration (min / mean / max): 0.05 / 1.14 / 2.43
  censored: 1 of 606 (0.2%)
  PROBLEM: first sample predates the discharge record


Both functions say what is wrong: 40 samples have no discharge for their
day. The fix is to obtain the missing discharge record or, failing that, to
keep only the samples it covers:

In [13]:
first_day = daily_short["Date"].min()
last_day = daily_short["Date"].max()
covered = (sample["Date"] >= first_day) & (sample["Date"] <= last_day)
sample_covered = sample[covered]
sample_covered = sample_covered.reset_index(drop=True)

elist_short = merge_report(info, daily_short, sample_covered)
overview_short = data_overview(elist_short)

Discharge record: 9769 days (26.7 years), 1985-01-01 to 2011-09-30
  Q (min / mean / max): 0.009911 / 4.125 / 246.4
Water-quality record: 566 samples (21.2 per year)
  1985-01-10 to 2011-09-29
  concentration (min / mean / max): 0.05 / 1.15 / 2.43
  censored: 1 of 566 (0.2%)


## 6. The period of analysis

Annual results are computed for a **period of analysis**. The default is the
water year, October through September, labelled by the calendar year in which
it ends. If the question is about one season, set the period with `set_pa`:
its first month (`pa_start`) and its length in months (`pa_long`). For
example, the growing season from April through August:

In [14]:
from pyegret.analysis import set_pa

elist_summer = set_pa(elist, pa_start=4, pa_long=5)
elist_summer["INFO"][["paStart", "paLong"]]

,paStart,paLong
0,4,5


Tables, plots and trend functions read the period from `INFO` from then on.
The model itself is the same whatever the period; only the way daily
estimates are gathered into annual results changes. A year is reported only
if the discharge record covers the whole period.

## 7. Before you fit: a checklist

- **Discharge**: daily, continuous, at least as long as the sampling, and in
  the units you told `read_user_daily` (cfs unless `q_unit=2`).
- **Concentration**: in mg/L, with non-detects marked `<` and given the
  reporting limit as their value.
- **Coverage**: `data_overview` lists no problems. Look at the last panel of
  `multi_plot_data_overview` (Tutorial 1) to see whether samples cover the
  whole range of flows.
- **Enough samples**: by default the model wants at least 100 samples, 50 of
  them uncensored, inside each weighting window, and widens the windows until
  it has them. A short or sparse record therefore gives a smoother, less local
  model. WRTDS was designed for long records sampled across all seasons and
  flows; two decades of monthly or more frequent sampling is typical of
  published applications.
- **Censoring**: a high share of censored values (`data_overview` flags more
  than half) weakens what the model can say, especially about change at low
  concentrations.

With the eList built, the analysis itself is the one in Tutorial 1, starting
from `model_estimation(elist)`.